# DataMate trên Colab Pro: toàn bộ pipeline với Qwen local

Chạy lần lượt từ trên xuống: clone repo → Qwen tạo manifest → ingestion toàn bộ train/public test → kiểm tra DuckDB → Qwen rewrite/sinh SQL → chấm train → xuất bài nộp.

Đầu vào là `dataset/train` và `dataset/public_test` trong repo clone. Không cần Gemini API key. Model được nạp một lần và dùng lại giữa các bước. Ingestion dùng Python/DuckDB, mọi tác vụ LLM dùng Qwen local. Manifest được sinh theo từng bảng nguồn, kiểm định schema, cột và bằng chứng; lỗi sẽ dừng bước chuẩn bị dữ liệu.

Chọn GPU trong **Runtime → Change runtime type**. `hf_quant="auto"` chọn lượng tử hóa theo VRAM; mức context vẫn cần kiểm tra thực tế. Kết quả mặc định ở `/content/datamate_sql_out`, bị mất khi Colab xóa runtime. Có thể đặt `OUT` trong Drive để giữ kết quả; khi đó notebook tự mount Drive.

In [11]:
from dataclasses import dataclass
from pathlib import Path
import base64
import getpass
import os
import subprocess
import sys

@dataclass
class ColabRuntime:
    repo_url: str = "https://github.com/Oszumeiii/ZAIC_2026.git"
    branch: str = "main"
    checkout_dir: str = "/content/ZAIC_2026"
    log_dir: str = "/content/datamate_runtime_logs"

    def checkout(self):
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = getpass.getpass("GitHub token cho repo private: ")
        if not token:
            raise RuntimeError("Thiếu GITHUB_TOKEN")
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        git = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]
        target = Path(self.checkout_dir)
        def run(args, cwd=None):
            result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
            output = (result.stdout + result.stderr).replace(token, "***").replace(auth, "***")
            print(output[-1500:])
            if result.returncode:
                raise RuntimeError("Git lỗi: kiểm tra token, nhánh và thay đổi trong checkout")
        if (target / ".git").is_dir():
            run(git + ["fetch", "origin", self.branch], target)
            run(["git", "checkout", self.branch], target)
            run(git + ["pull", "--ff-only", "origin", self.branch], target)
        else:
            run(git + ["clone", "--branch", self.branch, self.repo_url, str(target)])
        if not (target / "app/local_pipeline.py").is_file():
            raise RuntimeError("Nhánh clone chưa có app/local_pipeline.py; cần push bản full pipeline trước")
        self.root = target.resolve()
        os.chdir(self.root)
        if str(self.root) not in sys.path:
            sys.path.insert(0, str(self.root))
        commit = subprocess.run(["git", "log", "-1", "--oneline"], cwd=self.root,
                                check=True, capture_output=True, text=True)
        print(commit.stdout.strip())
        return self.root

    def run_logged(self, args, *, label, log_name, check=True, env=None):
        """Forward child stdout/stderr into the notebook and save the complete log."""
        log_path = Path(self.log_dir) / log_name
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"▶ {label} | log: {log_path}", flush=True)
        child_env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
        with log_path.open("w", encoding="utf-8") as log:
            with subprocess.Popen(
                args, cwd=self.root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, encoding="utf-8", errors="replace", bufsize=1, env=child_env,
            ) as process:
                try:
                    for line in process.stdout:
                        print(line, end="", flush=True)
                        log.write(line)
                        log.flush()
                    returncode = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                    raise
        if check and returncode:
            raise RuntimeError(f"{label} thất bại (exit={returncode}). Xem lỗi phía trên hoặc {log_path}")
        return returncode

    def install(self):
        self.run_logged(
            [sys.executable, "-m", "pip", "install", "-r", str(self.root / "requirements-colab.txt")],
            label="Cài thư viện", log_name="install.log")
        result = self.run_logged(
            [sys.executable, "-m", "pip", "install", "flash-linear-attention"],
            label="Cài kernel tùy chọn", log_name="flash-linear-attention.log", check=False)
        if result:
            print("flash-linear-attention không khả dụng; dùng implementation torch")

    def check_gpu(self):
        self.run_logged(["nvidia-smi"], label="Kiểm tra GPU", log_name="gpu.log")
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Cần runtime có CUDA GPU")
        print("torch", torch.__version__, "GPU", torch.cuda.get_device_name(0))

    def verify_manifest_runtime(self):
        """Reject an old checkout or stale imports before loading Qwen/ingesting data."""
        import ast
        import importlib
        source = self.root / "app/manifests/local.py"
        tree = ast.parse(source.read_text(encoding="utf-8"))
        definition = next(node for node in tree.body
                          if isinstance(node, ast.ClassDef) and node.name == "LocalManifestGenerator")
        disk_version = next(ast.literal_eval(node.value) for node in definition.body
                            if isinstance(node, ast.Assign)
                            and any(isinstance(target, ast.Name) and target.id == "VERSION" for target in node.targets))
        has_fix = any(isinstance(node, ast.FunctionDef) and node.name == "canonicalize_metadata"
                      for node in definition.body)
        print(f"Manifest trên đĩa: {disk_version} | {source}", flush=True)
        if not has_fix:
            raise RuntimeError(
                "Repo clone chưa có bản sửa manifest. Commit/push app/manifests/local.py và "
                "app/manifests/validation.py từ máy local, rồi chạy lại ô clone trên Colab.")
        module = importlib.import_module("app.manifests.local")
        orchestration = importlib.import_module("app.local_pipeline")
        loaded = module.LocalManifestGenerator
        bound = orchestration.DatasetPreparationPipeline.prepare.__globals__["LocalManifestGenerator"]
        print(f"Manifest trong RAM: {loaded.VERSION} | pipeline dùng: {bound.VERSION}", flush=True)
        if (Path(module.__file__).resolve() != source.resolve()
                or any(getattr(cls, "VERSION", None) != disk_version
                       or not hasattr(cls, "canonicalize_metadata") for cls in (loaded, bound))):
            raise RuntimeError(
                "File đã cập nhật nhưng Python vẫn giữ class manifest cũ. "
                "Chọn Runtime > Restart session rồi chạy lại các ô từ đầu; không cần xóa OUT.")
        print("Manifest metadata fix: OK", flush=True)

    def run_checks(self):
        self.verify_manifest_runtime()
        import importlib.metadata
        import json
        import platform
        import xml.etree.ElementTree as ET
        logs = Path(self.log_dir)
        logs.mkdir(parents=True, exist_ok=True)
        import hashlib
        revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=self.root,
                                  capture_output=True, text=True, check=True).stdout.strip()
        sql_source = self.root / "app/tools/sql.py"
        versions = {"python": sys.version, "platform": platform.platform(),
                    "git_commit": revision, "sql_executor": str(sql_source),
                    "sql_sha256": hashlib.sha256(sql_source.read_bytes()).hexdigest()}
        for name in ("pytest", "pandas", "numpy", "duckdb", "jsonschema", "openpyxl",
                     "transformers", "torch", "google-genai"):
            try:
                versions[name] = importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                versions[name] = "NOT INSTALLED"
        (logs / "environment.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
        print(json.dumps(versions, indent=2), flush=True)
        # Probe the actual checkout in a fresh process, independent of cached notebook imports.
        probe = "\n".join([
            "import duckdb",
            "from app.tools.sql import SQLExecutor",
            "with duckdb.connect() as connection:",
            "    try:",
            "        SQLExecutor.query(connection, 'PRAGMA version')",
            "    except ValueError:",
            "        print('SQL guard: OK — PRAGMA bị chặn')",
            "    else:",
            "        raise SystemExit('Checkout chưa có bản sửa PRAGMA. Commit/push app/tools/sql.py từ máy local, rồi chạy lại ô clone trên Colab.')",
        ])
        self.run_logged([sys.executable, "-c", probe], label="Xác nhận bản sửa SQL",
                        log_name="sql_guard.log")
        report = logs / "pytest.xml"
        report.unlink(missing_ok=True)
        tests = ["tests/test_local_pipeline.py", "tests/test_sql_pipeline.py",
                 "tests/test_architecture.py", "tests/test_business_manifest.py",
                 "tests/test_manifest_v2.py", "tests/test_main.py"]
        result = self.run_logged(
            [sys.executable, "-m", "pytest", "-q", "-ra", "--tb=short", "--color=no",
             f"--junitxml={report}", *tests],
            label="Kiểm tra pipeline local/SQL", log_name="pytest.log", check=False,
            env={"PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"})
        if result:
            if report.is_file():
                for case in ET.parse(report).iter("testcase"):
                    for kind in ("failure", "error"):
                        issue = case.find(kind)
                        if issue is not None:
                            print(f"{kind.upper()}: {case.get('classname')}.{case.get('name')}", flush=True)
                            print((issue.text or issue.get("message", ""))[-2000:], flush=True)
            raise RuntimeError(
                f"Pytest chưa đạt (exit={result}). Chi tiết: {logs / 'pytest.log'}; "
                f"phiên bản thư viện: {logs / 'environment.json'}. Pipeline chưa được khởi tạo.")
        print("Kiểm tra pipeline local/SQL: OK", flush=True)

    @staticmethod
    def prepare_output(out):
        if str(out).startswith("/content/drive/"):
            from google.colab import drive
            drive.mount("/content/drive")
        Path(out).mkdir(parents=True, exist_ok=True)

## 1. Clone và cài thư viện
Đặt `GITHUB_TOKEN` trong Colab Secrets, bật Notebook access. Nếu trước đó đã import transformers rồi đổi phiên bản thư viện, restart session và chạy lại từ đầu.

In [12]:
runtime = ColabRuntime()
ROOT = runtime.checkout()
runtime.install()
runtime.check_gpu()

From https://github.com/Oszumeiii/ZAIC_2026
 * branch            main       -> FETCH_HEAD
   23e3230..b020518  main       -> origin/main

Your branch is behind 'origin/main' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Already on 'main'

Updating 23e3230..b020518
Fast-forward
 notebooks/colab_pro.ipynb | 67 ++++++++++++++++++-----------------------------
 1 file changed, 26 insertions(+), 41 deletions(-)
From https://github.com/Oszumeiii/ZAIC_2026
 * branch            main       -> FETCH_HEAD

b020518 Fix DuckDB PRAGMA guard
▶ Cài thư viện | log: /content/datamate_runtime_logs/install.log
▶ Cài kernel tùy chọn | log: /content/datamate_runtime_logs/flash-linear-attention.log
▶ Kiểm tra GPU | log: /content/datamate_runtime_logs/gpu.log
Fri Nov 13 16:48:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     

## 2. Kiểm tra mã và cấu hình pipeline
Các test tích hợp SQL/local, manifest và CLI dùng dữ liệu/backend giả, không tải Qwen và không gọi Gemini. Đây là gate cho pipeline này; bộ test toàn repo còn lỗi có sẵn được ghi trong tài liệu. `OUT` giữ manifest, database, báo cáo và cache. Khi chạy lại, manifest chỉ tái sử dụng nếu khớp nguồn, schema và model; DuckDB chỉ tái sử dụng nếu fingerprint khớp.
Output của pytest được in trực tiếp vào ô và lưu tại `/content/datamate_runtime_logs/pytest.log`; phiên bản thư viện nằm trong `environment.json`. Nếu kiểm tra lỗi, notebook dừng và hiển thị tên test thất bại. Không bỏ qua kiểm tra để chạy tiếp.

Lỗi `PRAGMA version` trên DuckDB 1.5.6 đã được sửa ở `app/tools/sql.py`: kiểm tra từ khóa câu SQL gốc trước khi DuckDB phân tích. Sau khi cập nhật mã lên GitHub, chạy lại ô clone để lấy bản sửa rồi chạy ô kiểm tra này; giữ nguyên các test.


In [13]:
runtime.run_checks()

from app.local_pipeline import LocalPipelineConfig, LocalSQLPipeline

OUT = "/content/datamate_sql_out"
# Muốn giữ kết quả qua phiên: OUT = "/content/drive/MyDrive/datamate_sql_out"
runtime.prepare_output(OUT)
config = LocalPipelineConfig(
    dataset_dir=str(ROOT / "dataset"),
    out_dir=OUT,
    model="Qwen/Qwen3.5-9B",
    hf_quant="auto",
    max_model_len=16384,
    manifest_tokens=6144,
    sql_tokens=2048,
    n_samples=3,
    agree_min=2,
)
pipeline = LocalSQLPipeline(config)

{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "git_commit": "b020518cfc1c982bb525b244ba6b8cf38e388ced",
  "sql_executor": "/content/ZAIC_2026/app/tools/sql.py",
  "sql_sha256": "285ab30b8f558e86c3975c0d94bcb09dda5dc99cec01be59e1a9b0b723364c98",
  "pytest": "8.4.2",
  "pandas": "2.2.3",
  "numpy": "2.4.6",
  "duckdb": "1.5.6",
  "jsonschema": "4.26.0",
  "openpyxl": "3.1.5",
  "transformers": "5.22.1",
  "torch": "2.14.1+cu130",
  "google-genai": "2.12.1"
}
▶ Xác nhận bản sửa SQL | log: /content/datamate_runtime_logs/sql_guard.log
SQL guard: OK — PRAGMA bị chặn
▶ Kiểm tra pipeline local/SQL | log: /content/datamate_runtime_logs/pytest.log
.......................................................................  [100%]
-------- generated xml file: /content/datamate_runtime_logs/pytest.xml ---------
71 passed in 42.18s
Kiểm tra pipeline local/SQL: OK


## 3. Chuẩn bị toàn bộ dữ liệu trước agent
Bước này nạp Qwen lần đầu, tạo BusinessManifest v2 cho từng business, kiểm định, rồi dựng DuckDB với Raw/Typed/Semantic và catalog. Manifest đầu ra ở `OUT/manifests/<split>/`, database ở `OUT/databases/<split>/<business>/current/data.duckdb`. Không ghi manifest vào dataset gốc.

Chạy lại ô này để tiếp tục sau gián đoạn; các batch manifest hợp lệ đã lưu được dùng lại. Không chuyển sang agent khi bước này còn báo lỗi. Nếu prompt manifest vượt context, thông báo sẽ chỉ rõ; tăng `max_model_len` trong giới hạn VRAM và khởi tạo lại pipeline sau khi restart session.
Manifest generator `source-batches-v3` chuẩn hóa metadata phụ thuộc: `order_date → time_role=order_date`, `line_amount/base_quantity → aggregation=sum` khi model trả giá trị mâu thuẫn. Giá trị unknown/null được giữ nguyên; kiểm tra nguồn, bằng chứng và đơn vị tính vẫn bắt buộc. Mọi sửa đổi được ghi tại `OUT/manifests/<split>/reports/normalization/<business>/`. Lỗi cuối cùng được lưu tại `reports/failures/`.

Sau khi cập nhật mã từ GitHub, dùng **Runtime → Restart session** rồi chạy lại các ô từ đầu để tránh dùng class Python cũ đã nạp vào RAM. Không cần xóa dataset hoặc thư mục OUT.


In [14]:
runtime.verify_manifest_runtime()
preparation = pipeline.prepare_datasets()
validation = pipeline.validate_databases()

Tải Qwen/Qwen3.5-9B về ./models/Qwen__Qwen3.5-9B (cần bật Internet trong Settings của notebook) ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

[hf] /content/ZAIC_2026/models/Qwen__Qwen3.5-9B | GPU 42GB | nạp full | torch.bfloat16 | lai linear-attention: True


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[prepare/train] biz_101: Qwen manifest
Built input for biz_101: 1 files, 3 README sections, 1 survey results, 11 rulebook sections.
[manifest] biz_101 batch 1/1: ban_hang.csv / ban_hang


ValueError: Invalid manifest for biz_101/ban_hang: order_date has conflicting time_role=transaction_date; line_amount requires aggregation=sum; base_quantity requires aggregation=sum

## 4. Smoke test: 3 câu
Agent chỉ dùng database đã chuẩn bị và còn hợp lệ. Qwen chuẩn hóa câu hỏi, sinh JSON chứa SQL, nhận lỗi DuckDB để sửa và áp dụng đồng thuận. Xem lỗi sinh/SQL và VRAM trước khi chạy toàn bộ train.

In [ ]:
_, _, smoke_scores, _, smoke_logs = pipeline.smoke_test()
for row in smoke_logs:
    print(row.get("id"), row.get("type"), row.get("sql"), row.get("errors"))

## 5. Đánh giá toàn bộ train
Mặc định xử lý tất cả câu train, không giới hạn thời gian. Đáp án chuẩn chỉ dùng sau suy luận để chấm PA. Khi cần đo holdout: dùng `pipeline.evaluate(subset="holdout", holdout_n=6, per_biz=5)` thay lời gọi phía dưới. Thời gian toàn bộ train cần ước tính từ smoke test; chưa có số đo GPU thực tế trong notebook.

In [ ]:
_, _, train_scores, _, train_logs = pipeline.evaluate()
print(train_scores.status.value_counts())

## 6. Dự đoán toàn bộ public test và kiểm định bài nộp
Không giới hạn số câu/thời gian. Kết quả ở `OUT/public_test/submission.jsonl`, log kèm SQL và lỗi từng câu. Dùng lại cùng đối tượng `pipeline` nên không tải lại Qwen. Trước khi reset runtime, tải file hoặc sao lưu `OUT` nếu đang dùng ổ `/content`.

In [ ]:
# submission_path = pipeline.predict_public_test()
# print("Bài nộp:", submission_path)




# Tải về máy nếu cần:
# from google.colab import files
# files.download(str(submission_path))